In [1]:
import lmstudio as lms
import requests
import json
import os
import re
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

/home/nipdep/Dev/idea_graph/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pydantic import BaseModel, Field
from typing import Literal

In [3]:
DATA_PATH = "../twc_data/"
DOCLING_URL = "http://localhost:8080/documents/convert"

In [4]:
k = 500
ok = 0

In [5]:
def load_markdown(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    return text

def save_markdown(file_path, text):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

def save_xml(file_path, xml_text):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(xml_text)

In [6]:
def get_smart_chunks(text, max_chunk_size=500):
    """
    Yields chunks of text from a file based on complex splitting rules.

    1. Splits by "##" (keeping "##" at the start of the new chunk).
    2. If a "##" chunk is larger than max_chunk_size (in words),
    it splits that chunk by "\n\n".
    3. It then re-combines the "\n\n" pieces into sub-chunks that
    fit under the max_chunk_size.

    Args:
        file_path (str): The path to the text file.
        max_chunk_size (int): The target maximum number of words per chunk.
    """

    # Step 1: Split by "##" but keep the delimiter
    # re.split(r'(##)') splits the text but includes "##" in the list.
    # Example: "abc##def" -> ['abc', '##', 'def']
    splits = re.split(r'(##)', text)

    primary_chunks = []
    if splits:
        # Add the first part (before any "##") if it's not empty
        if splits[0].strip():
            primary_chunks.append(splits[0])
        
        # Combine "##" with the text that follows it
        for i in range(1, len(splits), 2):
            if i + 1 < len(splits):
                combined_chunk = splits[i] + splits[i+1] # "##" + "def"
                primary_chunks.append(combined_chunk)
            else:
                primary_chunks.append(splits[i]) # Handle trailing "##"

    # Step 2: Process each primary chunk
    for chunk in primary_chunks:
        chunk_words = chunk.split()
        word_count = len(chunk_words)

        # If the chunk is already fine, yield it
        if word_count <= max_chunk_size:
            if chunk.strip(): # Don't yield empty strings
                yield chunk
        
        # Step 3: If chunk is too big, sub-split by "\n\n"
        else:
            secondary_pieces = chunk.split('\n\n')
            current_sub_chunk_lines = []
            current_sub_chunk_word_count = 0

            for piece in secondary_pieces:
                piece_word_count = len(piece.split())

                # Check if this single piece is already over the limit
                if piece_word_count > max_chunk_size:
                    # If we have a sub-chunk built up, yield it first
                    if current_sub_chunk_lines:
                        yield "\n\n".join(current_sub_chunk_lines)
                        current_sub_chunk_lines = []
                        current_sub_chunk_word_count = 0
                    
                    # Yield the oversized piece by itself
                    yield piece
                    continue # Move to the next piece
                
                # If adding this piece would go over the limit, yield the current sub-chunk
                if current_sub_chunk_word_count + piece_word_count > max_chunk_size:
                    yield "\n\n".join(current_sub_chunk_lines)
                    
                    # Start a new sub-chunk with the current piece
                    current_sub_chunk_lines = [piece]
                    current_sub_chunk_word_count = piece_word_count
                
                # Otherwise, add this piece to the current sub-chunk
                else:
                    current_sub_chunk_lines.append(piece)
                    current_sub_chunk_word_count += piece_word_count
            
            # Yield any remaining sub-chunk after the loop finishes
            if current_sub_chunk_lines:
                yield "\n\n".join(current_sub_chunk_lines)



In [7]:
def iterative_spliter(text, chunk_size, chunk_overlap):
    splitter = CharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_text(text)


In [8]:
# future_task_instructions = """
# 2. Section Tags: this is semantic representation of the sections
#     - <abstract>: Represents the abstract of the document.
#     - <appendix>: Represents the appendix section.
#     - <background>: Represents the background section.
#     - <conclusion>: Represents the conclusion section.
#     - <contributions>: Represents the contributions section.
#     - <data>: Represents the data section.
#     - <discussion>: Represents the discussion section.
#     - <evaluation>: Represents the evaluation section.
#     - <future_work>: Represents the future work section.
#     - <introduction>: Represents the introduction section.
#     - <methodology>: Represents the methodology section.
#     - <model>: Represents the model section.
#     - <mortivation>: Represents the motivation section.
#     - <related_work>: Represents the related work section.
#     - <results>: Represents the results section.
    
# """



In [ ]:
def base_structure_prompt_builder(chunk, snapshot, with_head=False):
    instructions = """
    You will receive a markdown document chunk-by-chunk. For each chunk, analyze its structure and
    content, then convert the detected information into a **custom XML format** that captures the document’s
    hierarchy. Only create elements that correspond to real content found in the chunk.

    ====================
    ## 1. HIGH-LEVEL ELEMENTS
    These appear at most once in the full document, but individual chunks may contain them.
    Create the corresponding tag **only if the chunk actually contains the content**.

    - <title>  
        *The main title of the document.*  
        Usually appears at the beginning, but impurities (dates, journal names) may appear above it.  
        Extract the most likely true title and remove identifiable impurities.

    - <authors>  
        Contains one or more <author> elements.

    - <keywords>  
        Contains one or more <keyword> elements.

    - <references>  
        Contains one or more <reference> elements.  
        **Do NOT wrap <references> in a <section>.**

    - <section>  
        Represents a standard document section.  
        Can contain: <section-title>, <p>, <list>, <table>, <figure>.

    ====================
    ## 2. DOCUMENT COMPONENT ELEMENTS

    - <section-title>  
        Only created when the chunk has markdown headers: #, ##, ###, etc.

    - <p>  
        A paragraph inside a <section>.

    - <list>  
        A list (bullet or numbered).  
        **Do NOT structure list items.**  
        Wrap the raw list block inside <list>.

    - <table>  
        A table block.  
        **Do NOT structure rows or cells.**

    - <figure>  
        A figure **caption**.  
        If the text is only the image filename, skip it.

    - <author>  
        Contains the author name and email.

    - <keyword>  
        A single keyword.

    - <reference>  
        Structure:
            <reference>
                <ref-id>...</ref-id>        (Optional if present in markdown)
                <ref-authors>...</ref-authors>
                <ref-title>...</ref-title>
                <ref-venue>...</ref-venue>
            </reference>

    ====================
    ## 3. MUST-FOLLOW RULES

    ### Content Rules
    - Populate every tag only with actual markdown text.
    - Remove clearly identifiable impurities (dates, boilerplate, etc.).
    - Do NOT create elements that are not present in the chunk.
    - Do NOT duplicate information (e.g., placing the title again inside a section).

    ### Structural Rules
    - High-level elements (<title>, <authors>, <keywords>, <references>)  
    **must NOT be wrapped in <section>**.

    - BUT if a chunk contains at least one of those high-level elements,  
    **also create a <section> for any remaining non-high-level content** in the chunk.

    - Do NOT output an XML code block — return **raw XML only**.

    ### Chunking Rules
    - Process each chunk independently.  
    - Use “Last 200 tokens from previous chunk” only as context to avoid duplications,  
    **not** for generating new content.

    ====================
    ## 4. OUTPUT FORMAT

    Return **only** the XML elements detected in this chunk.  
    No explanations, no wrapper text, no ```xml fence.

    """

    head_structure = """
    <title>The Title of Your Paper</title>

    <authors>
        <author>
            <name>Jane Doe</name>
            <email>jane@something.com</email>
        </author>
        <author>
            <name>Anna Jane</name>
        </author>
    </authors>

    <keywords>
        <keyword>Large Language Model</keyword>
        <keyword>Information Extraction</keyword>
    </keywords>

    <venue>
        <name>Knowledge Graph Conference</name>
        <year>2024</year>
    </venue>
    """

    base_structure = """
    <section> 
        <section-title>1. Introduction</section-title>
        <p>This is the first paragraph of the introduction. It sets the stage for the paper.</p>
        <p>This is the second paragraph, providing more background information.</p>
        <list>
            - Bullet point one
            - Bullet point two
            - Bullet point three
        </list>
        <table>
            | Column 1 | Column 2 |
            |----------|----------|
            | Data 1   | Data 2   |
        </table>
        <figure>
            ![Figure 1: An illustrative figure](figure1.png)
            <caption>Figure 1: An illustrative figure</caption>
        </figure>
    </section>
    """

    if with_head:
        expected_structure = head_structure + base_structure
    else:
        expected_structure = base_structure

    prompt = f"""
    Task Instructions:
    {instructions}

    Previous Context (last 200 tokens):
    {snapshot}

    Current Chunk:
    {chunk}
    """

    return prompt

In [10]:

def base_structure_builder(text_chunk, llm_client, prompt_builder, history="", with_head=False):
    prompt = prompt_builder(chunk=text_chunk, snapshot=history, with_head=with_head)

    response = llm_client.respond(prompt, config={"temperature": 0.1})
    # print(response.content)
    return response.content 

In [11]:
# now the task is to detect discourse elements from the base structure. 

def discourse_structure_prompt_builder(chunk, snapshot, with_head=False):
    instructions = """
    You are given a <section> element extracted from a markdown document. Your task is to classify the section into one of the predefined categories based on following priority rules:
    1. If the <section> does not contain a section title (i.e., <section-title> tag), then that section must be classified as a subsection.
    2. Else, look at the section title to determine whether it matches one of the predefined categories.
    3. If the section title does not clearly match any predefined category, assess the content of the section to determine if it is a subsection of one of the categories or if additional context is needed for accurate classification.
    """  

    Example = """ 
    Input : 
    <section> 
        <section-title>1. Introduction</section-title>
        <p>This is the first paragraph of the introduction. It sets the stage for the paper.</p>
        <p>This is the second paragraph, providing more background information.</p>
        <list>
            - Bullet point one
            - Bullet point two
            - Bullet point three
        </list>
    </section>
    Output : 
    {
        "category": "introduction",
        "need_surrounding": false
    }

    Input :
    <section>
        <p>This section delves into the specifics of our proposed model architecture, detailing its components and functionalities.</p>
        <figure>
            ![Figure 2: Model Architecture](model_architecture.png)
        </figure>
    </section>
    Output :
    {
        "category": "subsection",
        "need_surrounding": true
    }
    """

    Task = """
    If the <section> does not contain a section title (i.e., <section-title> tag), then that section must be classified as a subsection.
        > {"category": "subsection", "need_surrounding": true}

    Else
        [Option-1] classify the given section into one of : 
            > "abstract"
            > "appendix"
            > "background"
            > "conclusion"
            > "contributions"
            > "data"
            > "discussion"
            > "evaluation"
            > "future_work"
            > "introduction"
            > "methodology"
            > "model"
            > "motivation"
            > "related_work"
            > "results"
            by setting the value of "class" key in the output json and "false" to the "need surrounding" key.

        [Option-2] Or you think the given section is a subsection of the above categories, then return
            > {"category": "subsection", "need_surrounding": false}.

        [Option-3] If and only if you think that you need surrounding sections to provide better context for classification return
            > {"category": "subsection", "need_surrounding": true}

    """

    prompt = f"""
        Detailed Instructions: {instructions} \n
        Example: {Example} \n
        Input : {chunk} \n
        Task : {Task} \n
    """

    return prompt



In [12]:
# 1. Define the literal type for the allowed class names
ClassCategory = Literal["subsection", "abstract", "appendix", "background", "conclusion", "contributions", "data", "discussion", "evaluation", "future_work", "introduction", "methodology", "model", "motivation", "related_work", "results"]

class SectionClassificationOutput(BaseModel):
    """
    Defines the expected output structure for the LLM classification task.
    """
    category: ClassCategory
    need_surrounding: bool

def discourse_structure_builder(section_chunk, llm_client, prompt_builder, output_structure):
    prompt = prompt_builder(chunk=section_chunk, snapshot="")

    response = llm_client.respond(prompt, config={"temperature": 0.2}, response_format=output_structure)

    # Parse the response content into the Pydantic model
    try:
        classification_output = response.parsed
    except Exception as e:
        response = llm_client.respond(prompt, config={"temperature": 0.2}, response_format=output_structure)
        classification_output = response.parsed

    # print(classification_output)
    return classification_output["category"]

In [13]:
def section_editor(section_text, new_section_class): 
    """This is a regex based section element update code. where we search for <section> and </section> tags, if the new_section_class is "subsection" then change the <section> -> <subsection> and </section> -> </subsection> if it's anoter class then just add that as a property <section class="new_section_class"> and </section> remains same. """
    if new_section_class == "subsection":
        # Change <section> to <subsection> and </section> to </subsection>
        updated_section = re.sub(r'<section>', '<subsection>', section_text)
        updated_section = re.sub(r'</section>', '</subsection>', updated_section)
    else:
        # Add class attribute to <section> tag
        updated_section = re.sub(r'<section>', f'<section class="{new_section_class}">', section_text)
    
    return updated_section


In [14]:

def markdown_formatter(text, structuring_llm_client, classification_llm_client, verbose=True):

    splits = list(get_smart_chunks(text, max_chunk_size=k))

    snap = ""
    xml_output = ""

    iterator = tqdm(splits, desc="Formatting chunks", unit="chunk", disable=not verbose)
    for i, chunk in enumerate(iterator):
        # print(f"--- Processing Chunk {i} ---")
        # print(chunk)
        # --- building base structure ---
        # if i == 0:
        #     chunk_xml = base_structure_builder(text_chunk=chunk, llm_client=structuring_llm_client, prompt_builder=base_structure_prompt_builder, history=snap, with_head=True)   
        # else:
        chunk_xml = base_structure_builder(text_chunk=chunk, llm_client=structuring_llm_client, prompt_builder=base_structure_prompt_builder, history=snap)
        
        # print("Base Structure XML: ", chunk_xml)

        # if the section is <abstract> or <references> then we skip the discourse classification and section editing step.
        if i != 0 and not re.search(r'<abstract>', chunk_xml) and not re.search(r'<references>', chunk_xml) and not re.search(r'<keywords>', chunk_xml) and not re.search(r'<authors>', chunk_xml) and not re.search(r'<title>', chunk_xml):
            # --- classifying sections --- 
            section_class_ = discourse_structure_builder(section_chunk=chunk_xml, llm_client=classification_llm_client, prompt_builder=discourse_structure_prompt_builder, output_structure=SectionClassificationOutput)

            # --- modifying chunk xml based on section class ---
            chunk_xml = section_editor(section_text=chunk_xml, new_section_class=section_class_)

        xml_output += chunk_xml + "\n"

        # Update snapshot with the last 200 tokens of the current chunk
        snap = ' '#.join(chunk.split()[-50:])

        if verbose:
            iterator.set_postfix({'chunks': len(xml_output.splitlines())})
            # print("Prompt: ", prompt)
            # print(chunk_xml)

    return xml_output

In [15]:

def xml_editor(xml_text): 
    # first we need to add <paper> root tag and </paper> end tag to make it a valid xml.
    xml_text = "<?xml version=\"1.0\" encoding=\"UTF-8\"?> \n<paper>\n" + xml_text + "\n</paper>"

    # second we need to replace & with &amp; to make it a valid xml.
    xml_text = xml_text.replace("&", "&amp;")

    return xml_text


In [ ]:
paper_id = "atgbfseomcs"
# --- load markdown file ---
md_text = load_markdown(DATA_PATH + paper_id + ".md")

# --- create clients ---
structuring_llm_client = lms.llm("qwen3-30b-a3b-instruct-2507")
classification_llm_client = lms.llm("openai/gpt-oss-120b")

# --- format markdown to structured XML --- 
formatted_xml = markdown_formatter(md_text, structuring_llm_client, classification_llm_client, verbose=True)


# --- save structured XML ---
formatted_xml = xml_editor(formatted_xml)
save_xml(DATA_PATH + paper_id + "_structured_new.xml", formatted_xml)

Formatting chunks:  83%|████████▎ | 40/48 [12:35<02:31, 18.89s/chunk, chunks=466]


LMStudioServerError: Chat response error: Reached context length of 5224 tokens, but this model does not currently support mid-generation context overflow because arch is gpt-oss. Try reloading with a larger context length or shortening the prompt/chat.

In [17]:

# text = load_markdown(DATA_PATH + "tog_converted.md")
# splits = list(get_smart_chunks(text, max_chunk_size=k))
# for i, chunk in enumerate(splits):
#     print(f"--- Chunk {i} ---")
#     print(chunk)
#     print("\n\n")